In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

PACK = Path("../data/raw/cairogo_pack").resolve()

trips_original = pd.read_csv(PACK / "trips.csv")
customers_original = pd.read_csv(PACK / "customers.csv")
drivers_original = pd.read_csv(PACK / "drivers.csv")
zones_original = pd.read_csv(PACK / "zones.csv")

trips = trips_original.copy(deep=True)
customers = customers_original.copy(deep=True)
drivers = drivers_original.copy(deep=True)
zones = zones_original.copy(deep=True)

datasets = {
    "trips": trips,
    "customers": customers,
    "drivers": drivers,
    "zones": zones
}

In [6]:
dtype_report = []

for name, df in datasets.items():

    for column in df.columns:

        dtype_report.append({
            "Dataset": name,
            "Column": column,
            "Current_Type": str(df[column].dtype),
            "Unique_Values": df[column].nunique(),
            "Missing_Values": df[column].isnull().sum()
        })

dtype_report = pd.DataFrame(dtype_report)

display(dtype_report)

,Dataset,Column,Current_Type,Unique_Values,Missing_Values
0,trips,trip_id,int64,27226,0
1,trips,customer_id,int64,5350,0
2,trips,driver_id,int64,1494,0
3,trips,request_ts,str,27177,0
4,trips,pickup_zone,str,24,0
5,trips,dropoff_zone,str,24,0
6,trips,distance_km,float64,2902,0
7,trips,surge_multiplier,float64,17,0
8,trips,est_fare,float64,2649,0
9,trips,eta_estimate_min,float64,233,0


In [7]:
for name, df in datasets.items():

    print("\n" + "=" * 60)
    print(f"{name.upper()} SAMPLE VALUES")
    print("=" * 60)

    for column in df.columns:
        print(f"{column}: {df[column].dropna().head(5).tolist()}")


TRIPS SAMPLE VALUES
trip_id: [100001, 100002, 100003, 100004, 100005]
customer_id: [21924, 20391, 24083, 25342, 25697]
driver_id: [5062, 5681, 5407, 5833, 6119]
request_ts: ['2026-05-01 00:01:46', '2026-05-01 00:37:36', '2026-05-01 00:40:12', '2026-05-01 00:49:23', '2026-05-01 01:05:33']
pickup_zone: ['Dokki', 'Dokki', 'Maadi', 'Dokki', 'Dokki']
dropoff_zone: ['Badr City', 'Obour', 'Obour', 'Mohandessin', 'Ramses']
distance_km: [17.22, 12.74, 10.8, 9.51, 4.54]
surge_multiplier: [1.1, 1.1, 1.0, 1.2, 1.0]
est_fare: [107.4, 114.9, 92.3, 75.5, 39.3]
eta_estimate_min: [6.4, 7.3, 6.4, 8.1, 8.6]
weather: ['rain', 'rain', 'rain', 'rain', 'rain']
payment_method: ['card', 'cash', 'card', 'cash', 'cash']
service_level: ['economy', 'comfort', 'comfort', 'economy', 'economy']
request_failed: [0, 0, 0, 0, 0]
actual_trip_time_min: [29.9, 28.9, 22.5, 28.9, 20.6]
cancel_reason: ['driver_rejected', 'no_driver_found', 'no_driver_found', 'driver_cancelled', 'driver_rejected']
support_ticket_flag: [0, 0, 

In [11]:
id_columns = {
    "trips": ["trip_id", "customer_id", "driver_id"],
    "customers": ["customer_id"],
    "drivers": ["driver_id"],
    "zones": ["zone_id"]
}

for name, columns in id_columns.items():

    df = datasets[name]

    for col in columns:
        df[col] = df[col].astype("string")


date_columns = {
    "trips": ["request_ts"],
    "customers": ["signup_date"],
    "drivers": ["join_date"]
}

date_conversion_results = []

for name, columns in date_columns.items():

    df = datasets[name]

    for col in columns:

        original_values = df[col].copy()

        df[col] = pd.to_datetime(
            original_values,
            errors="coerce"
        )

        invalid_count = (
            original_values.notna() & df[col].isna()
        ).sum()

        date_conversion_results.append({
            "Dataset": name,
            "Column": col,
            "Invalid_Dates": invalid_count,
            "Converted_Type": str(df[col].dtype)
        })

for name, df in datasets.items():

    print("\n" + "=" * 50)
    print(name.upper())
    print("=" * 50)

    print(df.dtypes)


TRIPS
trip_id                         string
customer_id                     string
driver_id                       string
request_ts              datetime64[us]
pickup_zone                        str
dropoff_zone                       str
distance_km                    float64
surge_multiplier               float64
est_fare                       float64
eta_estimate_min               float64
weather                            str
payment_method                     str
service_level                      str
request_failed                   int64
actual_trip_time_min           float64
cancel_reason                      str
support_ticket_flag              int64
driver_arrival_min             float64
dtype: object

CUSTOMERS
customer_id             string
signup_date     datetime64[us]
channel                    str
home_zone                  str
loyalty_tier               str
dtype: object

DRIVERS
driver_id                    string
join_date            datetime64[us]
vehicle_type    

In [14]:
exact_duplicates = []

for name, df in datasets.items():

    duplicate_count = df.duplicated().sum()

    exact_duplicates.append({
        "Dataset": name,
        "Total_Rows": len(df),
        "Exact_Duplicates": duplicate_count,
        "Duplicate_Percentage": round(
            duplicate_count / len(df) * 100, 2
        )
    })

exact_duplicates_report = pd.DataFrame(exact_duplicates)

display(exact_duplicates_report)

for name, df in datasets.items():

    duplicates = df[
        df.duplicated(keep=False)
    ]

    if not duplicates.empty:

        print(f"\n{'='*50}")
        print(f"EXACT DUPLICATES IN {name.upper()}")
        print("="*50)

        display(
            duplicates.sort_values(
                by=list(df.columns)
            )
        )

    else:
        print(f"{name}: No exact duplicate rows found.")

,Dataset,Total_Rows,Exact_Duplicates,Duplicate_Percentage
0,trips,27226,0,0.0
1,customers,6000,0,0.0
2,drivers,1500,0,0.0
3,zones,24,0,0.0


In [16]:
primary_keys = {
    "trips": "trip_id",
    "customers": "customer_id",
    "drivers": "driver_id",
    "zones": "zone_id"
}

duplicate_id_results = []

for name, key in primary_keys.items():

    df = datasets[name]

    duplicate_count = df[key].duplicated().sum()

    duplicate_id_results.append({
        "Dataset": name,
        "Primary_Key": key,
        "Total_Rows": len(df),
        "Unique_IDs": df[key].nunique(),
        "Duplicate_ID_Records": duplicate_count
    })

duplicate_id_report = pd.DataFrame(
    duplicate_id_results
)

display(duplicate_id_report)

for name, key in primary_keys.items():

    df = datasets[name]

    duplicate_ids = df.loc[
        df[key].duplicated(keep=False)
    ].sort_values(by=key)

    if not duplicate_ids.empty:

        print(f"\nDuplicate IDs in {name.upper()}:")

        display(duplicate_ids)

    else:
        print(f"{name}: No duplicate primary keys.")

,Dataset,Primary_Key,Total_Rows,Unique_IDs,Duplicate_ID_Records
0,trips,trip_id,27226,27226,0
1,customers,customer_id,6000,6000,0
2,drivers,driver_id,1500,1500,0
3,zones,zone_id,24,24,0


trips: No duplicate primary keys.
customers: No duplicate primary keys.
drivers: No duplicate primary keys.
zones: No duplicate primary keys.


In [18]:
customer_ids = set(customers["customer_id"].dropna())

trip_customer_ids = set(trips["customer_id"].dropna())

missing_customer_ids = trip_customer_ids - customer_ids

print("Customer IDs referenced in trips:",
      len(trip_customer_ids))

print("Customer IDs not found in customers:",
      len(missing_customer_ids))

print("Sample missing IDs:",
      list(missing_customer_ids)[:10])

driver_ids = set(drivers["driver_id"].dropna())

trip_driver_ids = set(trips["driver_id"].dropna())

missing_driver_ids = trip_driver_ids - driver_ids

print("Driver IDs referenced in trips:",
      len(trip_driver_ids))

print("Driver IDs not found in drivers:",
      len(missing_driver_ids))

print("Sample missing IDs:",
      list(missing_driver_ids)[:10])

valid_zones = set(zones["zone_name"].dropna())

pickup_zones = set(trips["pickup_zone"].dropna())
dropoff_zones = set(trips["dropoff_zone"].dropna())

invalid_pickup_zones = pickup_zones - valid_zones
invalid_dropoff_zones = dropoff_zones - valid_zones

print("Invalid pickup zones:",
      invalid_pickup_zones)

print("Invalid drop-off zones:",
      invalid_dropoff_zones)

Customer IDs referenced in trips: 5350
Customer IDs not found in customers: 0
Sample missing IDs: []
Driver IDs referenced in trips: 1494
Driver IDs not found in drivers: 0
Sample missing IDs: []
Invalid pickup zones: {'Badr City', 'Shubra'}
Invalid drop-off zones: {'Badr City', 'Shubra'}


In [20]:
display(
    zones[
        zones["zone_name"].str.contains(
            "Badr|Shubra",
            case=False,
            na=False
        )
        |
        zones["legacy_name"].str.contains(
            "Badr|Shubra",
            case=False,
            na=False
        )
    ]
)

valid_zone_names = set(zones["zone_name"].dropna())
valid_legacy_names = set(zones["legacy_name"].dropna())

for zone in ["Badr City", "Shubra"]:

    print(f"\nZone: {zone}")
    print("In zone_name:", zone in valid_zone_names)
    print("In legacy_name:", zone in valid_legacy_names)


valid_zone_names = set(zones["zone_name"].dropna())
valid_legacy_names = set(zones["legacy_name"].dropna())

valid_zone_names_and_legacy = (
    valid_zone_names | valid_legacy_names
)

invalid_pickup = (
    set(trips["pickup_zone"].dropna())
    - valid_zone_names_and_legacy
)

invalid_dropoff = (
    set(trips["dropoff_zone"].dropna())
    - valid_zone_names_and_legacy
)

print("Unmatched pickup zones:", invalid_pickup)
print("Unmatched dropoff zones:", invalid_dropoff)

,zone_id,zone_name,legacy_name,zone_type,has_metro
4,Z05,Shubra District,Shubra,downtown,1
20,Z21,Badr City District,Badr City,suburb,0



Zone: Badr City
In zone_name: False
In legacy_name: True

Zone: Shubra
In zone_name: False
In legacy_name: True
Unmatched pickup zones: set()
Unmatched dropoff zones: set()


In [21]:
valid_zone_names = (
    set(zones["zone_name"].dropna())
    | set(zones["legacy_name"].dropna())
)

zone_validation = pd.DataFrame([
    {
        "Column": "pickup_zone",
        "Unique_Values": trips["pickup_zone"].nunique(),
        "Unmatched_Values": len(
            set(trips["pickup_zone"].dropna()) - valid_zone_names
        ),
        "Status": "Passed" if not (
            set(trips["pickup_zone"].dropna()) - valid_zone_names
        ) else "Failed"
    },
    {
        "Column": "dropoff_zone",
        "Unique_Values": trips["dropoff_zone"].nunique(),
        "Unmatched_Values": len(
            set(trips["dropoff_zone"].dropna()) - valid_zone_names
        ),
        "Status": "Passed" if not (
            set(trips["dropoff_zone"].dropna()) - valid_zone_names
        ) else "Failed"
    }
])

display(zone_validation)

,Column,Unique_Values,Unmatched_Values,Status
0,pickup_zone,24,0,Passed
1,dropoff_zone,24,0,Passed


In [24]:
numerical_columns = {
    "trips": [
        "distance_km",
        "surge_multiplier",
        "est_fare",
        "eta_estimate_min",
        "actual_trip_time_min",
        "driver_arrival_min"
    ],
    "drivers": [
        "experience_months",
        "rating_avg"
    ]
}

for name, columns in numerical_columns.items():

    print(f"\n{'='*60}")
    print(f"{name.upper()} NUMERICAL SUMMARY")
    print("="*60)

    display(
        datasets[name][columns].describe().T
    )



TRIPS NUMERICAL SUMMARY


,count,mean,std,min,25%,50%,75%,max
distance_km,27226.0,10.894424,6.678165,1.14,5.5,9.43,14.620,60.0
surge_multiplier,27226.0,1.257849,0.331323,1.00,1.0,1.10,1.400,2.6
est_fare,27226.0,103.337751,56.722721,27.20,65.4,90.60,124.300,718.0
eta_estimate_min,27226.0,9.015158,4.021844,2.70,5.8,8.20,11.475,29.4
actual_trip_time_min,21560.0,31.912426,12.326115,8.80,22.6,29.30,39.000,101.4
driver_arrival_min,21560.0,9.579875,4.372893,1.00,6.3,9.00,12.400,31.4



DRIVERS NUMERICAL SUMMARY


,count,mean,std,min,25%,50%,75%,max
experience_months,1500.0,38.520000,26.365502,1.00,19.00,32.00,52.00,140.0
rating_avg,1238.0,4.546502,0.184513,3.92,4.42,4.54,4.66,5.0


In [25]:
validation_rules = {
    "trips": {
        "distance_km": lambda s: s <= 0,
        "surge_multiplier": lambda s: s < 1,
        "est_fare": lambda s: s <= 0,
        "eta_estimate_min": lambda s: s <= 0,
        "actual_trip_time_min": lambda s: s <= 0,
        "driver_arrival_min": lambda s: s <= 0
    },
    "drivers": {
        "experience_months": lambda s: s < 0,
        "rating_avg": lambda s: (s < 1) | (s > 5)
    }
}

validation_results = []

for dataset_name, columns in validation_rules.items():
    df = datasets[dataset_name]

    for column, rule in columns.items():
        invalid_mask = rule(df[column])

        validation_results.append({
            "Dataset": dataset_name,
            "Column": column,
            "Invalid_Values": int(invalid_mask.sum()),
            "Status": "Passed" if invalid_mask.sum() == 0 else "Needs Investigation"
        })

validation_report = pd.DataFrame(validation_results)

display(validation_report)

,Dataset,Column,Invalid_Values,Status
0,trips,distance_km,0,Passed
1,trips,surge_multiplier,0,Passed
2,trips,est_fare,0,Passed
3,trips,eta_estimate_min,0,Passed
4,trips,actual_trip_time_min,0,Passed
5,trips,driver_arrival_min,0,Passed
6,drivers,experience_months,0,Passed
7,drivers,rating_avg,0,Passed
